# Full Model CPT - Ultra-Aggressive Memory Optimization
## Qwen2.5-7B - No Quantization, No LoRA, Pure bfloat16

- FULL model training
- No adapters/LoRA
- bfloat16 (no quantization)
- Minimal batch + sequence length
- Maximum memory efficiency

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

import torch
import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*80)
print("FULL MODEL CPT - ULTRA-AGGRESSIVE MEMORY OPTIMIZATION")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("Memory optimizations: ENABLED")
print("="*80 + "\n")

In [ ]:
# ============ ULTRA-MINIMAL CONFIG ============
config = {
    "model_name": "Qwen2.5-7B",
    "train_file": "pretraining_augmented_data/train.jsonl",
    "eval_file": "pretraining_augmented_data/eval.jsonl",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,      # ABSOLUTE MINIMUM
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 4,      # REDUCED from 8
    "learning_rate": 2e-5,
    "warmup_steps": 100,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "max_seq_length": 256,                 # REDUCED from 512
    "bf16": True,
    "output_dir": "medical_qwen_cpt_full",
    "save_steps": 50,
    "eval_steps": 25,
    "logging_steps": 5,
    "dataloader_num_workers": 0,
    "dataloader_pin_memory": False,
    "seed": 42,
}

print("Ultra-Aggressive Configuration:")
print("="*70)
print(f"Batch size: {config['per_device_train_batch_size']}")
print(f"Gradient accumulation: {config['gradient_accumulation_steps']}")
print(f"Effective batch: {config['per_device_train_batch_size'] * config['gradient_accumulation_steps']}")
print(f"Sequence length: {config['max_seq_length']}")
print(f"Precision: bfloat16 (full model, NO quantization)")
print(f"Gradient checkpointing: WILL BE ENABLED")
print("="*70 + "\n")

In [ ]:
# ============ LOAD DATA ============
def load_jsonl(file_path: str) -> List[Dict]:
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            try:
                data.append(json.loads(line))
            except:
                pass
    return data

print("Loading data...")
train_data = load_jsonl(config["train_file"])
eval_data = load_jsonl(config["eval_file"])

print(f"✅ Train: {len(train_data):,} chunks")
print(f"✅ Eval:  {len(eval_data):,} chunks\n")

In [ ]:
# ============ LOAD TOKENIZER ============
from transformers import AutoTokenizer

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    config["model_name"],
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded\n")

In [ ]:
# ============ LOAD MODEL (FULL, NO QUANTIZATION) ============
from transformers import AutoModelForCausalLM

print("Loading model (bfloat16, full precision)...")
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model = AutoModelForCausalLM.from_pretrained(
    config["model_name"],
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# CRITICAL: Enable gradient checkpointing
model.gradient_checkpointing_enable()

print(f"✅ Model loaded")
num_params = sum(p.numel() for p in model.parameters())
print(f"   Parameters: {num_params/1e9:.2f}B")
print(f"   Dtype: {next(model.parameters()).dtype}")
print(f"   Gradient checkpointing: ENABLED")

allocated = torch.cuda.memory_allocated(0) / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
free = total - allocated

print(f"\n   GPU allocated: {allocated:.2f} / {total:.2f} GB")
print(f"   GPU free: {free:.2f} GB")

if free < 15:
    print(f"\n⚠️  WARNING: Only {free:.2f}GB free - risky!")
else:
    print(f"\n✅ Good: {free:.2f}GB free\n")

In [ ]:
# ============ TOKENIZE ============
from datasets import Dataset

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=config["max_seq_length"],
        padding="max_length",
    )

print("Tokenizing...")
train_dataset = Dataset.from_dict({"text": [c["text"] for c in train_data]})
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

eval_dataset = Dataset.from_dict({"text": [c["text"] for c in eval_data]})
eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

print(f"✅ Train: {len(train_dataset):,} samples")
print(f"✅ Eval: {len(eval_dataset):,} samples\n")

In [ ]:
# ============ SETUP TRAINER ============
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir=config["output_dir"],
    num_train_epochs=config["num_train_epochs"],
    per_device_train_batch_size=config["per_device_train_batch_size"],
    per_device_eval_batch_size=config["per_device_eval_batch_size"],
    gradient_accumulation_steps=config["gradient_accumulation_steps"],
    learning_rate=config["learning_rate"],
    warmup_steps=config["warmup_steps"],
    weight_decay=config["weight_decay"],
    max_grad_norm=config["max_grad_norm"],
    bf16=config["bf16"],
    save_strategy="steps",
    save_steps=config["save_steps"],
    save_total_limit=1,
    eval_strategy="steps",
    eval_steps=config["eval_steps"],
    logging_dir=config["logging_dir"],
    logging_steps=config["logging_steps"],
    seed=config["seed"],
    dataloader_num_workers=config["dataloader_num_workers"],
    dataloader_pin_memory=config["dataloader_pin_memory"],
    load_best_model_at_end=True,
    greater_is_better=False,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("✅ Trainer ready\n")
print("="*80)
print("🚀 STARTING FULL MODEL TRAINING")
print("="*80)

In [ ]:
# ============ TRAIN ============
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    train_result = trainer.train()
    print(f"\n✅ Complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Loss: {train_result.training_loss:.4f}\n")
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ============ EVAL & SAVE ============
print("Evaluating...")
eval_results = trainer.evaluate()
print(f"Eval loss: {eval_results.get('eval_loss', 'N/A'):.4f}")

best_model_path = Path(config["output_dir"]) / "best_model"
best_model_path.mkdir(parents=True, exist_ok=True)

print(f"\nSaving to {best_model_path}...")
trainer.model.save_pretrained(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

print("✅ DONE")